# 03 — DNN TabNet

Notebook ini mengimplementasikan TabNet untuk data tabular credit risk.
TabNet digunakan sebagai model DNN karena memiliki attention mechanism untuk memilih fitur penting pada setiap decision step.

Output utama:
- Training dengan early stopping `patience=20`
- Plot training loss curve
- Evaluasi Accuracy, Precision, Recall, F1-score, dan AUC-ROC
- Perbandingan TabNet dengan model terbaik traditional ML
- Simpan model ke `models/tabnet_model.zip`

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
DATA_RAW = PROJECT_ROOT / "data" / "raw" / "credit_risk_dataset.csv"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from preprocessing import run_full_pipeline


## 1. Load data hasil preprocessing

TabNet memakai data numerik hasil encoding dan scaling dari pipeline preprocessing.

In [ ]:
processed_files = ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]
missing_processed = [name for name in processed_files if not (DATA_PROCESSED / name).exists()]

if missing_processed:
    print("Processed files belum lengkap, menjalankan run_full_pipeline()...")
    X_train, X_test, y_train, y_test = run_full_pipeline(str(DATA_RAW))
else:
    X_train = pd.read_csv(DATA_PROCESSED / "X_train.csv")
    X_test = pd.read_csv(DATA_PROCESSED / "X_test.csv")
    y_train = pd.read_csv(DATA_PROCESSED / "y_train.csv").squeeze("columns")
    y_test = pd.read_csv(DATA_PROCESSED / "y_test.csv").squeeze("columns")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train distribution:", y_train.value_counts().to_dict())
print("y_test distribution :", y_test.value_counts().to_dict())


## 2. Import TabNet

Jika import gagal, install dependency dengan `pip install pytorch-tabnet` atau jalankan ulang `pip install -r requirements.txt`.

In [ ]:
try:
    import torch
    from pytorch_tabnet.tab_model import TabNetClassifier
except ImportError as exc:
    raise ImportError(
        "pytorch-tabnet belum terinstall. Jalankan: pip install pytorch-tabnet"
    ) from exc


## 3. Siapkan train/validation/test set

Data training sudah diseimbangkan dengan SMOTE dari pipeline. Sebagian training set dipisah menjadi validation set untuk early stopping.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train,
)

X_tr_np = X_tr.values.astype(np.float32)
X_val_np = X_val.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)

y_tr_np = y_tr.values.astype(np.int64)
y_val_np = y_val.values.astype(np.int64)
y_test_np = y_test.values.astype(np.int64)

print("Train:", X_tr_np.shape)
print("Valid:", X_val_np.shape)
print("Test :", X_test_np.shape)


## 4. Definisi model TabNet

In [ ]:
model = TabNetClassifier(
    n_d=32,
    n_a=32,
    n_steps=5,
    gamma=1.5,
    n_independent=2,
    n_shared=2,
    momentum=0.02,
    mask_type="entmax",
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-3, weight_decay=1e-5),
    scheduler_params={"step_size": 10, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    seed=42,
    verbose=10,
)


## 5. Training dengan early stopping

In [ ]:
model.fit(
    X_train=X_tr_np,
    y_train=y_tr_np,
    eval_set=[(X_tr_np, y_tr_np), (X_val_np, y_val_np)],
    eval_name=["train", "valid"],
    eval_metric=["auc"],
    max_epochs=200,
    patience=20,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False,
)


## 6. Plot training loss curve

In [ ]:
history = model.history

plt.figure(figsize=(10, 5))
plt.plot(history["loss"])
plt.title("TabNet Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

if "valid_auc" in history:
    plt.figure(figsize=(10, 5))
    plt.plot(history["valid_auc"])
    plt.title("TabNet Validation AUC")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.tight_layout()
    plt.show()


## 7. Evaluasi TabNet

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

y_pred = model.predict(X_test_np)
y_proba = model.predict_proba(X_test_np)[:, 1]

tabnet_metrics = {
    "Model": "TabNet",
    "Accuracy": accuracy_score(y_test_np, y_pred),
    "Precision": precision_score(y_test_np, y_pred, zero_division=0),
    "Recall": recall_score(y_test_np, y_pred, zero_division=0),
    "F1": f1_score(y_test_np, y_pred, zero_division=0),
    "AUC-ROC": roc_auc_score(y_test_np, y_proba),
}

print(pd.Series(tabnet_metrics).to_string())
print("
Classification report:")
print(classification_report(y_test_np, y_pred, target_names=["Good", "Default"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test_np,
    y_pred,
    display_labels=["Good", "Default"],
    cmap="Blues",
    values_format="d",
)
plt.title("Confusion Matrix — TabNet")
plt.show()


## 8. Bandingkan TabNet vs model terbaik traditional ML

Jika notebook `02_traditional_ml.ipynb` sudah dijalankan, file metadata model terbaik akan dibaca otomatis.

In [ ]:
comparison_rows = [tabnet_metrics]
metadata_path = MODELS_DIR / "best_model_metadata.pkl"
results_path = MODELS_DIR / "traditional_ml_results.csv"

if metadata_path.exists():
    traditional_metadata = joblib.load(metadata_path)
    traditional_row = {
        "Model": f"Best Traditional ML ({traditional_metadata['best_model_name']})",
        **traditional_metadata["metrics"],
    }
    comparison_rows.insert(0, traditional_row)
elif results_path.exists():
    traditional_results = pd.read_csv(results_path)
    best_traditional = traditional_results.sort_values(["F1", "AUC-ROC"], ascending=False).iloc[0]
    traditional_row = {
        "Model": f"Best Traditional ML ({best_traditional['Model']})",
        "Accuracy": best_traditional["Accuracy"],
        "Precision": best_traditional["Precision"],
        "Recall": best_traditional["Recall"],
        "F1": best_traditional["F1"],
        "AUC-ROC": best_traditional["AUC-ROC"],
    }
    comparison_rows.insert(0, traditional_row)
else:
    print("File hasil traditional ML belum ada. Jalankan notebooks/02_traditional_ml.ipynb dulu untuk perbandingan lengkap.")

comparison_table = pd.DataFrame(comparison_rows)
comparison_table


## 9. Simpan TabNet model

TabNet akan tersimpan sebagai `models/tabnet_model.zip`.

In [ ]:
model.save_model(str(MODELS_DIR / "tabnet_model"))
joblib.dump(tabnet_metrics, MODELS_DIR / "tabnet_metrics.pkl")

print("Saved:")
print(MODELS_DIR / "tabnet_model.zip")
print(MODELS_DIR / "tabnet_metrics.pkl")
